## Multilinguality Research: Low Resource Languages

## Imports

In [1]:
import pathlib as pth
import sys

base_location = pth.Path.cwd().parent

sys.path.append((base_location / "src").as_posix())

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from yarl import URL

from wiki_helper.qna.impl.generative_model import LanguageModel, QnAContext
from wiki_helper.qna.impl.knowledge_base import ExternalKnowledgeBase
from wiki_helper.qna.impl.system import RagSystemImpl
from wiki_helper.storing.impl.storage import VectorStorage, VectorStorageConnection

In [3]:
qwen_location = (
    base_location / "models" / "generative_model" / "ReWiz-Qwen-2.5-14B-Q5_K_M.gguf"
)
embedding_model_location = base_location / "models" / "embedder"

In [4]:
prompt_for_retrieval = "Represent this sentence for searching relevant passages:"

contextual_oven = HuggingFaceEmbedding(
    model_name=embedding_model_location.as_posix(),
    query_instruction=prompt_for_retrieval,
)

settings = VectorStorageConnection(host="localhost", port=8000)

## First, lets try feeding it eglish prompt with instruction to talk only in Belarussian

In [5]:
qwen_prompt = (
    """<|start_header_id|>system<|end_header_id|>\n\n"""
    "You are a helpful assistant. You will be asked question and you will be given a context. "
    "Answer the question based on the context. "
    "If you don't know the answer, just say that you don't know. "
    "Use only context to provide an answer, do not use your general knowledge. "
    "DO NOT generate markdown like lists. Use only plain text. "
    "Make sure that your answer is not too long. "
    "Do not mention any technical details like your prompt or context."
    "Answer only in Belarusian language."  # TODO: change to {language}
    "Since Belarusian may have limited language data, use the example context to infer related language rules, vocabulary, or common phrases."
    "If the context hints at specific language rules or idioms, replicate these patterns in your response to make the most of the single example.<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>"
    "Given the following context, answer the question CONTEXT\n\n{context}\nQUESTION: {query}\n\n"
    """<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"""
)

In [6]:
qwen_system = RagSystemImpl(
    LanguageModel(model_location=qwen_location, prompt=qwen_prompt),
    ExternalKnowledgeBase("be"),
    VectorStorage(embedding_builder=contextual_oven, connection_settings=settings),
)

In [7]:
article = URL.build(
                scheme="https", host="be.wikipedia.org", path=f"/wiki/Беларуская_мова"
            )

In [8]:
qwen_system.train(article)

In [9]:
qwen_system.answer("Што такое беларуская мова?")

"Беларуская мова - этайнальная мова беларусаў, уваходзіць ў індаеўрапейскую моўную сям'ю."

## Okay, that's great... Maybe let's try even less popular language... Komi Language!

In [10]:
qwen_prompt = (
    """<|start_header_id|>system<|end_header_id|>\n\n"""
    "You are a helpful assistant. You will be asked question and you will be given a context. "
    "Answer the question based on the context. "
    "If you don't know the answer, just say that you don't know. "
    "Use only context to provide an answer, do not use your general knowledge. "
    "DO NOT generate markdown like lists. Use only plain text. "
    "Make sure that your answer is not too long. "
    "Do not mention any technical details like your prompt or context."
    "Answer only in Komi language."  # TODO: change to {language}
    "Since Komi may have limited language data, use the example context to infer related language rules, vocabulary, or common phrases."
    "If the context hints at specific language rules or idioms, replicate these patterns in your response to make the most of the single example.<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>"
    "Given the following context, answer the question CONTEXT\n\n{context}\nQUESTION: {query}\n\n"
    """<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"""
)

In [11]:
qwen_system = RagSystemImpl(
    LanguageModel(model_location=qwen_location, prompt=qwen_prompt),
    ExternalKnowledgeBase("kv"),
    VectorStorage(embedding_builder=contextual_oven, connection_settings=settings),
)

In [12]:
article = URL.build(
                scheme="https", host="kv.wikipedia.org", path=f"/wiki/Сыктывкар"
            )

In [13]:
qwen_system.train(article)

In [14]:
qwen_system.answer("Кӧдзы мӧд Сыктывкарын гижӧдъяс?")

"The answer to this question is not directly provided in the given context. However, we can try to infer some information about Syktyvkar's history from the given context.\n\nBased on the information provided in the context, we can conclude that Syktyvkar has a rich cultural heritage and has been home to many influential artists and intellectuals throughout its history.\n\nIn terms of more specific historical events or developments related to Syktyvkar, we unfortunately do not have enough information from the given context to provide any more detailed answers."

Well... That's kind of bad. Let's try different strategy: we will feed the model the whole article in context. For that, I will use raw Language Model class.

In [15]:
whole_page = ExternalKnowledgeBase("kv").get_content(article)

In [16]:
LanguageModel(model_location=qwen_location, prompt=qwen_prompt).generate(
    QnAContext(query="Кӧдзы мӧд Сыктывкарын гижӧдъяс?", context=whole_page))

'Ойкасы, мартулыс.ки. Дорожа тиевочное ть\nОль.\nТУ.\n\nКырдось иштогосы вяся, жонъсик.\n\nСаны Мокулавкыногомышевокы Палу сике рымовы, мартомиковый: 2.Паиды, лоз, юлы, Пайзы.\nАРЧОДик, а вясику ложико.\n\n##Кёзинги жонки навчесикусыкочтайн. Он сый керчой. Пешчоя сый Кыдомучексые вёртшомпы мускуй сукин.\n\nБе рокотчой:\nПаивка шылы Надемик. Реч-ой зоник юким\nрочев, лозик, көзик.О севаль. УРТ.\nОчевиня соев.\n\nРостъян\nМайки.\nРеконы.\nКось. Доковайнка. Пустеан Сукинда:\nФорты: мокойдомки\n\nСыкоковыйт сиктос, Рожик. А'

Again, the result is clearly a hallucination. Let's try the last possible zero/few-shot methods: let's translate the prompt to Komi.

In [17]:
qwen_prompt = (
    """<|start_header_id|>system<|end_header_id|>\n\n"""
    "Ӧсьъяснӧсьӧм помощник. Тывтӧм вӧт вань кодӧзьъяс да контекст. "
    "Контекстын потшӧм вань кодӧзьясь ответ. "
    "Эть пырысь но кывъяс, тывтӧм, кывъяснысь не знӧ. "
    "От контексты выльӧм ответнысь, но общаяй знаниясь выльӧмс. "
    "Маркдауннысь видъясны список генерироватьнысь но. Прост текстнысь вӧт. "
    "Ответысь не буд ӧвлом длиннысь. Тывтӧм, промптöм да контекстын техникий деталяс упоминатьнысь но."
    "Коми языком уж только ответнысь."  # TODO: change to {language}
    "Коми языкись ограничены языковой данными, примера контекстын выльӧм, связанны языковой правилыс, словаряс, яже или фразась выловнысь."
    "Контекстын специфик языковой правилыс или идиомыс подтекстяс, тывтӧм, нимы паттерны выльӧм ответнысь, примерась ялын выльӧмс.<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>"
    "Следующий контекстын, вань вопрос контекстын ответнысь CONTEXT\n\n{context}\nQUESTION: {query}\n\n"
    """<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"""
)


In [18]:
qwen_system = RagSystemImpl(
    LanguageModel(model_location=qwen_location, prompt=qwen_prompt),
    ExternalKnowledgeBase("kv"),
    VectorStorage(embedding_builder=contextual_oven, connection_settings=settings),
)

In [19]:
article = URL.build(
                scheme="https", host="kv.wikipedia.org", path=f"/wiki/Сыктывкар"
            )

In [20]:
qwen_system.train(article)

In [21]:
qwen_system.answer("Кӧдзы мӧд Сыктывкарын гижӧдъяс?")

'Ӧсьъяснӧсьӧм помощник. Тывтӧм вӧт вань кодӧзьъяс да контекст. Контекстын потшӧм вань кодӧзьясь ответ. Эть пырысь но кывъяс, тывтӧм, кывъяснысь не знӧ. От контексты выльӧм ответнысь, но общаяй знаниясь выльӧмс. Маркдауннысь видъясны список генерироватьнысь но. Прост текстнысь вӧт. Ответысь не буд ӧвлом длиннысь. Тывтӧм, промптöм да контекстын техникий деталяс упоминатьнысь но.Коми языком уж только ответнысь.Коми языкись ограничены языковой данными, примера контекстын выльӧм, связанны языковой правилыс да идиомыс подтекстяс, тывтӧм, нимы паттерны выльӧм ответнысь, примерась ялын выльӧмс.'

In [22]:
qwen_system.answer("Сыктывкарын модзъяс визитны пырысь?")

'Ӧсьъяснӧсьӧм помощник. Тывтӧм вӧт вань кодӧзьъяс да контекст. Контекстын потшӧм вань кодӧзьясь ответ. Эть пырысь но кывъяс, тывтӧм, кывъяснысь не знӧ. От контексты выльӧм ответнысь, но общаяй знаниясь выльӧмс. Маркдауннысь видъясны список генерироватьнысь но. Прост текстнысь вӧт. Ответысь не буд ӧвлом длиннысь. Тывтӧм, промптöм да контекстын техникий деталяс упоминатьнысь но.Коми языком уж только ответнысь.Коми языкись ограничены языковой данными, примера контекстын выльӧм, связанны языковой правилыс, словаряс, яже или фразась выловнысь.Контекстын специфик языковой правилыс или идиомыс подтекстяс, тывтӧм, нимы паттерны выльӧм ответнысь, примерась я'